Hypothesis: Responses that are incorrect will have at least one active sae feature in the response region that has a high feature_context_specificity_score that is not present in the prompt region ("False features")
Responses that are correct will have SAE features that are better aligned between prompt and response

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import sys
import html as html_lib
from IPython.display import HTML, display as ipy_display
import matplotlib.pyplot as plt
from datasets import load_dataset

root = Path.cwd().resolve()
root = root.parent
sys.path.insert(0, str(root))

import torch
from collections import defaultdict
from transformers import AutoTokenizer, AutoModelForCausalLM
from sae_trainer.core.model_utils import SparseAutoencoder, ReluSparseAutoencoder, TopKSparseAutoencoder
from sae_trainer.sae_semantic_eval.feature_tracer_utils import FeatureTracer, TraceConfig
from sae_trainer.sae_semantic_eval.feature_tracer_eval_utils import build_feature_persistence_df, top_n_feats_by_hits_count, top_n_feats_by_act_mass, prompt_x_feature_heatmap, feature_token_position_profile, top_m_feature_coactivation, render_feature_card, plot_feature_umap, plot_inter_feature_similarity
from sae_trainer.sae_semantic_eval.feature_tracer_eval_utils import plot_prompt_vs_response_features, render_feature_card_comparison, plot_prompt_response_activation_scatter, render_prompt_response_token_card



In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
layer_idx = 12
dataset_name = 'openwebtext'
model_name_simple = "qwen" # Either gpt2 or qwen
sae_type = 'topk'
k = 40

if model_name_simple == "gpt2":
    model_name = "gpt2"
elif model_name_simple == "qwen":
    model_name = "Qwen/Qwen2-0.5B-Instruct"
else:
    raise ValueError(f"Invalid model name: {model_name_simple}. Expected either 'gpt2' or 'qwen'.")
tokenizer = AutoTokenizer.from_pretrained(model_name)
llm = AutoModelForCausalLM.from_pretrained(model_name).to(device).eval()

# Load SAE checkpoint — pick whichever layer you want to analyse (3, 6, 9, or 11)
ckpt = torch.load(root / f"model_weights_files/{sae_type}_sae_{model_name_simple}_{dataset_name}_layer{layer_idx}.pt", map_location=device)
if sae_type == 'relu':
    sae = ReluSparseAutoencoder(d_in=ckpt["d_in"], d_latent=ckpt["d_latent"], normalize_decoder=True).to(device)
elif sae_type == 'topk':
    sae = TopKSparseAutoencoder(d_in=ckpt["d_in"], d_latent=ckpt["d_latent"], k=k, normalize_decoder=True).to(device)
else:
    raise ValueError(f"Invalid sae type: {sae_type}. Expected either 'relu' or 'qwen'.")
sae.load_state_dict(ckpt["model_state"])
sae.eval()

TopKSparseAutoencoder(
  (encoder): Linear(in_features=896, out_features=7168, bias=True)
  (decoder): Linear(in_features=7168, out_features=896, bias=False)
)

In [7]:
# 4) Tracer config — layer_idx MUST match the checkpoint loaded above
prompt_tracer_cfg = TraceConfig(
    layer_idx=layer_idx,
    topk_per_token=8,
    min_activation=0.01,
    max_new_tokens=200,
    do_sample=True,
    temperature=0.7,
    token_mode='prompt_only',
    min_prompts=10
)
prompt_tracer = FeatureTracer(llm, tokenizer, sae, device, prompt_tracer_cfg)

# 4) Tracer config — layer_idx MUST match the checkpoint loaded above
response_tracer_cfg = TraceConfig(
    layer_idx=layer_idx,
    topk_per_token=8,
    min_activation=0.01,
    max_new_tokens=200,
    do_sample=True,
    temperature=0.7,
    token_mode='generated_only',
    min_prompts=10
)
response_tracer = FeatureTracer(llm, tokenizer, sae, device, response_tracer_cfg)

In [8]:
max_prompts = 100
min_words = 50
truncation_limit = 150

ds = load_dataset("trivia_qa", "rc.nocontext", split="validation", streaming=True)

# Collect prompts once
prompts, ids = [], []
for i, row in enumerate(ds):
    question = row["question"].strip()
    if len(question.split()) >= min_words:
        prompts.append(" ".join(question.split()[:truncation_limit]))
        ids.append(str(len(prompts) - 1))
    if len(prompts) >= max_prompts:
        break



In [9]:
# Both tracers see exactly the same prompts
prompt_summaries = prompt_tracer.trace_prompts(prompts, ids=ids)
response_summaries = response_tracer.trace_prompts(prompts, ids=ids)

In [10]:
# Score all features (calls compute_feature_embeddings internally)
prompt_scores = prompt_tracer.feature_specificity_scores()
response_scores = response_tracer.feature_specificity_scores()

In [12]:
# Build the full matrix (no filters — cheap to re-filter later)
persistence_df = build_feature_persistence_df(prompt_tracer, response_tracer)

# Filter to context-specific, low-token-specificity features with strong cross-tracer alignment
interesting = build_feature_persistence_df(
    prompt_tracer, response_tracer,
    min_context_specificity_vs_baseline=0.025,
    max_token_specificity_vs_baseline=0.04,
    min_embedding_sim=0.6,
)

# Features that persist most consistently across prompts
interesting.groupby("feature_id").agg(
    n_prompts=("prompt_id", "nunique"),
    mean_prompt_act=("prompt_activation", "mean"),
    mean_response_act=("response_activation", "mean"),
    embedding_sim=("embedding_sim", "first"),
).sort_values("n_prompts", ascending=False)

# Per-prompt profile: which "interesting" features fired on prompt 13?
interesting[interesting["prompt_id"] == "13"]

,prompt_id,feature_id,prompt_activation,response_activation,embedding_sim,context_specificity_vs_baseline,token_specificity_vs_baseline
57,13,273,2.187155,2.250577,0.756969,0.034161,-0.011037
58,13,341,2.663901,2.821020,0.816769,0.038783,-0.020144
59,13,726,2.587013,2.843858,0.689089,0.037850,-0.079470
60,13,792,2.758908,2.796157,0.814556,0.064203,-0.038641
61,13,859,3.067655,3.181920,0.995832,0.040867,0.032707
62,13,1453,2.924838,2.106501,0.987021,0.033297,0.038612
63,13,1562,3.904069,3.194917,0.850696,0.025640,-0.001211
64,13,1784,2.485601,3.217179,0.905360,0.046071,-0.007023
65,13,2311,2.444206,2.449702,0.868204,0.040041,0.029809
66,13,2393,2.538502,2.142831,0.697978,0.063635,0.010365


In [23]:
render_prompt_response_token_card(feature_id=5050, prompt_id=13, prompt_tracer=prompt_tracer, response_tracer=response_tracer)
